### Window Generation 

Window generation and data preparation for model ingestion. Benign and attack traffic are windowed into two separate sets. Each message is turned into a sliding window over consecutive messages. 

**Windows for Benign** would have window size: 32 and stride: 16 due to the volume of data. We will be ingesting three main files which contain 10M rows each. 

**Windows for Attack** would have window size: 32 and stride: 2 the attack files are smaller and the injection time is shorter, a small stride produces more windows for the model. 

The feature extractor should take a feature array with the shape [num_windows, window_size, num_features] and a target array with the shape [num_windows]. The return value should be the transformed feature and target arrays. 

Structure should follow (num_windows, window_size, features). 

#### Feature Vector 

Every message, benign or attack, is converted into the same 11-feature vector. Keeping one shared representation lets the model learn each attack's behaviour from the same inputs, and matches the features examined in the Attack EDA and Benign EDA. 

11 Features contain: [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ] 

- **CAN_ID** — identifier as an integer
- **b0–b7** — payload bytes, padded to 8 bytes
- **DLC** — data length code
- **Δt** — inter-arrival time since the previous message


In [ ]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd
import torch

BASE = Path("..").resolve()
sys.path.append(str(BASE / "src"))
from extract_can_fields import extract_can_fields 
from load_attack_file import load_attack_file, ATTACKS

In [ ]:
BENIGN_LOGS = BASE / "Data" / "CAN_MIRGU_Benign_Logs"

BENIGN_FILES = [
    BENIGN_LOGS / "Benign" / "Day_3" / "Benign_day3_file1.log",
    BENIGN_LOGS / "Benign 2" / "Day_5" / "Benign_day5_file1.log",
    BENIGN_LOGS / "Benign 2" / "Day_6" / "Benign_day6_file1.log",
]

WINDOW_SIZE = 32
STRIDE = 16
TRAIN_RATIO = 0.70

BENIGN_OUTPUT_DIR = BASE / "Data" / "Benign_Windows"
BENIGN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Benign files:", len(BENIGN_FILES))
print("Window size / stride:", WINDOW_SIZE, "/", STRIDE)
print("Train ratio:", TRAIN_RATIO)
print("Output directory:", BENIGN_OUTPUT_DIR)

Benign files: 3
Window size / stride: 32 / 16
Train ratio: 0.7
Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/Benign_windows


In [ ]:
ATTACK_WINDOW_SIZE = 32
ATTACK_STRIDE = 2
ATTACK_FILES = list(ATTACKS.keys())

ATTACK_OUTPUT_DIR = BASE / "Data" / "Attack_Windows"
ATTACK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Attack files:", len(ATTACK_FILES))
print(ATTACK_FILES)
print("Output directory:", ATTACK_OUTPUT_DIR)

Attack files: 9
['Steering_angle_attack', 'Brake_warning_attack', 'Power_steering_attack', 'Min_speedometer_attack_1', 'EMS_replay_attack', 'Steering_angle_replay', 'Fuzzing_random_IDs', 'Fuzzing_valid_IDs', 'DoS_attack']
Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/New_Attack_W


Process the message and extract the features to convert it into the 11 feature vector [CAN_ID] + 8 payload bytes + [DLC, Δt] = 11 features

In [4]:
def message_to_features(extracted, previous_timestamp): 
    can_id = int(extracted["can_id"], 16)
    
    payload_bytes = []
    for byte in extracted["payload"]:
        payload_bytes.append(int(byte, 16))
        
    timestamp = extracted["timestamp"]
    if previous_timestamp is None:
        delta_t = 0  # since the first message has no previous timestamp
    else:
        delta_t = timestamp - previous_timestamp
        
    features = [can_id] + payload_bytes + [extracted["dlc"], delta_t]
    return features, timestamp     

#### Processing Files

In [5]:
def load_message_features(file_path): 
    message_features = []
    previous_timestamp = None
    
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            extracted = extract_can_fields(line)
            if extracted is None:
                continue

            features, previous_timestamp = message_to_features(extracted, previous_timestamp)
            message_features.append(features)
            
    return message_features 

In [6]:
def make_windows(message_features, window_size,stride):
    windows = []
    current_window = []
    for feature in message_features:
        current_window.append(feature)
        if len(current_window) == window_size:
            windows.append(current_window.copy())
            current_window = current_window[stride:]
            
    return torch.tensor(windows, dtype=torch.float32)

Three files will be used for the creation of the benign windows, which then will be stored as .pt in a folder on the root of the project. 

1. Benign_day3_file1.log 
2. Benign_day5_file1.log 
3. Benign_day6_file1.log 

Benign files are windowed separately, file by file. They are not concatenated before windowing, otherwise the end of one recording could share a window with the start of another, and the delta time between those messages would not be real bus timing. 

The 12 benign files behave very similarly, since they all contain the same 56 CAN IDs, message rates, and share the same frequent CAN IDs in almost identical proportions. This shows that the benign traffic is stable across the different recording days. 

Therefore the selected files are not more representative than the others, however they contain enough messages for reliable analysis and come from three different days, reducing the risk of analysing behaviour that is specific to one recording session. 

Every window is labelled Benign since there is no injection interval. 

In [ ]:
for file_path in BENIGN_FILES: 
    print("Processing File: ", file_path.name)
    
    message_features = load_message_features(file_path)
    
    # Split by time: first 70% for training the SSL model, last 30% for checking reconstruction
    split_index = int(len(message_features) * TRAIN_RATIO)
    
    train_windows = make_windows(message_features[:split_index], WINDOW_SIZE, STRIDE)
    test_windows = make_windows(message_features[split_index:], WINDOW_SIZE, STRIDE)
    
    train_path = BENIGN_OUTPUT_DIR / f"{file_path.stem}_train.pt"
    test_path = BENIGN_OUTPUT_DIR / f"{file_path.stem}_test.pt"
    
    # Every benign window is class 0, so labels are all zeros 
    torch.save({"features": train_windows, "labels": torch.zeros(len(train_windows), dtype=torch.long)}, train_path)
    torch.save({"features": test_windows, "labels": torch.zeros(len(test_windows), dtype=torch.long)}, test_path)
    
    print("train:", train_windows.shape, "| test:", test_windows.shape)
    print("Saved:", train_path.name, "and", test_path.name)
    
    
print("Done.")

Benign .pt files store labels=0 for consistency with attack windows. SSL pretraining uses only features and does not use these labels.

Because the split happens before windowing, no window can contain messages from both sides, preventing data leakage. 

We obtain **6 .pt files: 3 for pretraining**, **3 for checking reconstruction**. Notebook 04 will load the _train.pt files to train the SSL model, and the _test.pt files only to measure reconstruction loss.

In [ ]:
pt_files = sorted(BENIGN_OUTPUT_DIR.glob("*_train.pt")) + sorted(BENIGN_OUTPUT_DIR.glob("*_test.pt"))
print("Train/test files found:", len(pt_files))

rows = []
for pt_path in pt_files:
    data = torch.load(pt_path, map_location="cpu", weights_only=False)

    features = data["features"]
    labels = data["labels"]
    split_name = "train" if pt_path.name.endswith("_train.pt") else "test"

    rows.append({
        "file": pt_path.name,
        "split": split_name,
        "n_windows": features.shape[0],
        "window_shape": tuple(features.shape[1:]),
        "label_values": labels.unique().tolist(),
    })
pd.DataFrame(rows)

Train/test files found: 6


,file,split,n_windows,window_shape,label_values
0,Benign_day3_file1_train.pt,train,429074,"(32, 11)",[0]
1,Benign_day5_file1_train.pt,train,456547,"(32, 11)",[0]
2,Benign_day6_file1_train.pt,train,449616,"(32, 11)",[0]
3,Benign_day3_file1_test.pt,test,183888,"(32, 11)",[0]
4,Benign_day5_file1_test.pt,test,195662,"(32, 11)",[0]
5,Benign_day6_file1_test.pt,test,192692,"(32, 11)",[0]


The table proves each window has the 32 x 11 feature vector, with label values 0 representing benign, and the train files hold the 70% and the test files 30% matching the split. 

##### Feature structure checks on saved windows

Each window is (32, 11): [CAN_ID, b0…b7, DLC, Δt].

- CAN ID ≤ 2047 (11-bit)
- payload bytes b0–b7 in 0–255
- DLC in 0–8
- Δt ≥ 0


In [ ]:
pt_files = sorted(BENIGN_OUTPUT_DIR.glob("*_train.pt")) + sorted(BENIGN_OUTPUT_DIR.glob("*_test.pt"))
print("Benign window structure checks")
for pt_path in pt_files:
    features = torch.load(pt_path, map_location="cpu", weights_only=False)["features"]
    can_ids = features[:, :, 0]
    payload = features[:, :, 1:9]
    dlc = features[:, :, 9]
    delta_t = features[:, :, 10]

    can_id_ok = bool(can_ids.max() <= 2047)
    payload_ok = bool(payload.min() >= 0 and payload.max() <= 255)
    dlc_ok = bool(dlc.min() >= 0 and dlc.max() <= 8)
    timing_ok = bool(delta_t.min() >= 0)

    print(pt_path.name)
    print("  CAN ID ≤ 2047:", can_id_ok, "| max:", int(can_ids.max()))
    print("  payload 0-255:", payload_ok)
    print("  DLC 0-8:", dlc_ok)
    print("  Δt ≥ 0:", timing_ok)


Benign window structure checks
Benign_day3_file1_train.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True
Benign_day5_file1_train.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True
Benign_day6_file1_train.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True
Benign_day3_file1_test.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True
Benign_day5_file1_test.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True
Benign_day6_file1_test.pt
  CAN ID ≤ 2047: True | max: 1535
  payload 0–255: True
  DLC 0–8: True
  Δt ≥ 0: True


### Attack Windows 

Attack windows use the same 11 feature vector as benign traffic [ CAN_ID , b0, b1, b2, b3, b4, b5, b6, b7, DLC, Δt ]. 

Each attack recording stays separate: windows are created per source file, never across files. Mixing files would break the time order of the bus.

Message labels come from load_attack_file: rows with flag=1 get the attack class; rows with flag=0 stay Benign. The injection interval is only used for timing analysis in the EDA, not for window labels.

A window is kept as Attack if it contains **at least one** message with flag=1. Windows with all flag=0 are discarded. The window label is the attack class of that file (DoS / Spoofing / Fuzzing / Replay).

Saved tensors go to the Attack_windows/ folder, with the corresponding .pt window files saved.

In [4]:
rows = []

for name in ATTACK_FILES: 
    df = load_attack_file(name)
    flags = df["flag"].tolist()
    
    total_windows = 0  # Total number of windows
    mixed_windows = 0  # Some flags are 0, some are 1
    pure_attack_windows = 0   # All flags are 1 
    pure_benign_windows = 0  # All flags are 0
    
    # Iterate over the message labels in steps of ATTACK_STRIDE
    for start in range(0, len(flags) - ATTACK_WINDOW_SIZE + 1, ATTACK_STRIDE):
        window_flags = flags[start : start + ATTACK_WINDOW_SIZE]
        # Count the total number of windows
        total_windows += 1 
        
        injected_count = sum(window_flags)
        
        # Check if the window contains both attack and benign messages
        if injected_count == 0: 
            pure_benign_windows += 1 
        elif injected_count == ATTACK_WINDOW_SIZE:
            pure_attack_windows += 1 
        else: 
            mixed_windows += 1 
        
    total_attack_windows = pure_attack_windows + mixed_windows      
    rows.append({
        "file": name, 
        "total_windows": total_windows, 
        "mixed_windows": mixed_windows, 
        "pure_attack_windows": pure_attack_windows, 
        "pure_benign_windows": pure_benign_windows, 
        "total_attack_windows": total_attack_windows,
        "percentage_mixed": round(100 * total_attack_windows / total_windows, 2),
    })

In [5]:
rows_df = pd.DataFrame(rows)
rows_df.head(10)

,file,total_windows,mixed_windows,pure_attack_windows,pure_benign_windows,total_attack_windows,percentage_mixed
0,Steering_angle_attack,184800,63110,0,121690,63110,34.15
1,Brake_warning_attack,294281,109950,0,184331,109950,37.36
2,Power_steering_attack,180584,49712,0,130872,49712,27.53
3,Min_speedometer_attack_1,272787,120336,0,152451,120336,44.11
4,EMS_replay_attack,162058,64393,0,97665,64393,39.73
5,Steering_angle_replay,237889,101481,0,136408,101481,42.66
6,Fuzzing_random_IDs,364488,120733,0,243755,120733,33.12
7,Fuzzing_valid_IDs,128691,22907,0,105784,22907,17.80
8,DoS_attack,170878,88094,0,82784,88094,51.55


- total windows: all attack windows 
- mixed windows: some flag=1 and some flag=0 (still labelled Attack)
- pure attack: windows that contain exclusively attack messages (all flags are 1), this column shows zeros since the attack recordings follow benign background traffic while the malicious messages are being injected into the CAN stream, so there is no sequence with 32 messages which contain only messages with 1. 

- pure benign: windows where all messages are benign (all flags are 0), which can be dropped from the attack dataset.
- percentage mixed: the total percentage of useful attack windows (combining pure attack and mixed windows) kept for training. 

### Flag Based Window Label

Attack windows are labelled 0–3 (DoS, Spoofing, Fuzzing, Replay). Windows with no flag=1 are dropped. 

In [6]:

CLASS_TO_ID = {
    "DoS": 0,
    "Spoofing": 1,
    "Fuzzing": 2,
    "Replay": 3,
}

print("Output directory:", ATTACK_OUTPUT_DIR)
print("Class mapping:", CLASS_TO_ID)


Output directory: /Users/anita/Documents/TFM/SSL_CyberSecurity/New_Attack_W
Class mapping: {'DoS': 0, 'Spoofing': 1, 'Fuzzing': 2, 'Replay': 3}


#### Processing Files 

In [7]:
def make_attack_windows(name, window_size, stride):
    df = load_attack_file(name)
    
    features = []
    for i in range(len(df)):
        # Convert the payload to a list of integers
        payload_bytes = [int(b, 16) for b in df["payload"].iloc[i]]
        # Create a feature vector for the current message
        features.append([df["id_int"].iloc[i]] + payload_bytes + [df["dlc"].iloc[i], df["dt"].iloc[i]])

    flags = df["flag"].tolist()
    class_id = CLASS_TO_ID[ATTACKS[name]]

    windows, window_labels = [], []
    
    # Iterate over the features in steps of the stride
    for start in range(0, len(features) - window_size + 1, stride):
        # If the window contains no attack messages, skip it
        if sum(flags[start:start + window_size]) < 1:
            continue 
        # Append the window and its label to the lists
        windows.append(features[start:start + window_size])
        window_labels.append(class_id)
        
    return (torch.tensor(windows, dtype=torch.float32), torch.tensor(window_labels, dtype=torch.long),
)

In [8]:
for name in ATTACK_FILES:
    print("Processing file:", name)

    windows, labels = make_attack_windows(name, ATTACK_WINDOW_SIZE, ATTACK_STRIDE)

    save_path = ATTACK_OUTPUT_DIR / f"{name}.pt"
    torch.save({"features": windows, "labels": labels}, save_path)

    print("Saved window file:", save_path.name)

print("All attack windows have been saved.")

Processing file: Steering_angle_attack
Saved window file: Steering_angle_attack.pt
Processing file: Brake_warning_attack
Saved window file: Brake_warning_attack.pt
Processing file: Power_steering_attack
Saved window file: Power_steering_attack.pt
Processing file: Min_speedometer_attack_1
Saved window file: Min_speedometer_attack_1.pt
Processing file: EMS_replay_attack
Saved window file: EMS_replay_attack.pt
Processing file: Steering_angle_replay
Saved window file: Steering_angle_replay.pt
Processing file: Fuzzing_random_IDs
Saved window file: Fuzzing_random_IDs.pt
Processing file: Fuzzing_valid_IDs
Saved window file: Fuzzing_valid_IDs.pt
Processing file: DoS_attack
Saved window file: DoS_attack.pt
All attack windows have been saved.


In [9]:
pt_files = sorted(ATTACK_OUTPUT_DIR.glob("*.pt"))
print("Files found:", len(pt_files))

rows = []
for pt_path in pt_files:
    data = torch.load(pt_path, map_location="cpu")

    features = data["features"]
    labels = data["labels"]

    rows.append({
        "file": pt_path.name,
        "keys": list(data.keys()),
        "features_shape": tuple(features.shape),
        "labels_shape": tuple(labels.shape),
        "dtype": str(features.dtype),
        "label_values": labels.unique().tolist(),
    })
    
pd.DataFrame(rows)

Files found: 9


,file,keys,features_shape,labels_shape,dtype,label_values
0,Brake_warning_attack.pt,"[features, labels]","(109950, 32, 11)","(109950,)",torch.float32,[1]
1,DoS_attack.pt,"[features, labels]","(88094, 32, 11)","(88094,)",torch.float32,[0]
2,EMS_replay_attack.pt,"[features, labels]","(64393, 32, 11)","(64393,)",torch.float32,[3]
3,Fuzzing_random_IDs.pt,"[features, labels]","(120733, 32, 11)","(120733,)",torch.float32,[2]
4,Fuzzing_valid_IDs.pt,"[features, labels]","(22907, 32, 11)","(22907,)",torch.float32,[2]
5,Min_speedometer_attack_1.pt,"[features, labels]","(120336, 32, 11)","(120336,)",torch.float32,[1]
6,Power_steering_attack.pt,"[features, labels]","(49712, 32, 11)","(49712,)",torch.float32,[1]
7,Steering_angle_attack.pt,"[features, labels]","(63110, 32, 11)","(63110,)",torch.float32,[1]
8,Steering_angle_replay.pt,"[features, labels]","(101481, 32, 11)","(101481,)",torch.float32,[3]
